# AmsterdamUMCdb — Person Table EDA

## Objectives for this notebook
- Characterise the patient population in AmsterdamUMCdb (v1.5.0, OMOP CDM)
- Quantify missingness in demographic fields
- Visualise gender, age, and birth-year distributions
- Identify quality issues that affect cohort selection

### Imports

In [ ]:
import pandas as pd
import numpy as np
from google.cloud import bigquery
import matplotlib.pyplot as plt

### Retrieving Google Project Id

In [ ]:
client = bigquery.Client()

print("Connected to BigQuery")

In [ ]:
# sets *your* project id
PROJECT_ID = "capstoneweaningprediction" #@param {type:"string"}

# Sets the default BigQuery dataset for accessing AmsterdamUMCdb

If you have received instructions to use a specific BigQuery instance, change the default settings here. Otherwise use these default values.

In [ ]:
# sets default dataset for AmsterdamUMCdb
DATASET_PROJECT_ID = 'amsterdamumcdb' #@param {type:"string"}
DATASET_ID = 'version1_5_0' #@param {type:"string"}
LOCATION = 'eu' #@param {type:"string"}

# Provide your credentials to access the AmsterdamUMCdb dataset on Google BigQuery
Authenticate your credentials with Google Cloud Platform and set your default Google Cloud Project ID as an environment variable for running query jobs.

1. Run the cell. The `Allow this notebook to access your Google credentials?` prompt appears. Select `Allow`.
2. In the `Sign in - Google Accounts` dialog, use the account you registered during the AmsterdamUMCdb application process and select `Allow` again.

In [ ]:
import os
from google.colab import auth

# all libraries check this environment variable, so set it:
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID

auth.authenticate_user()
print('Authenticated')

# Enable data table display

Colab includes the `google.colab.data_table` package that can be used to display Pandas dataframes as an interactive data table (default limits: `max_rows = 20000`, `max_columns = 20`). This is especially useful when exploring the  tables or dictionary from AmsterdamUMCdb. It can be enabled with:

In [ ]:
%load_ext google.colab.data_table
from google.colab.data_table import DataTable

# change default limits:
DataTable.max_columns = 50
DataTable.max_rows = 80000


## Set the default query job configuration for magics

In [ ]:
%load_ext bigquery_magics
from bigquery_magics import bigquery_magics
from google.cloud import bigquery

# sets the default query job configuration
def_config = bigquery.job.QueryJobConfig(default_dataset=DATASET_PROJECT_ID + "." + DATASET_ID)
bigquery_magics.context.default_query_job_config = def_config

## Query the `person` table and copy the data to the `persons` Pandas dataframe:

The `person` table contains a record for each patient in AmsterdamUMCdb.

Since this is a relatively small table, it is acceptable to use `SELECT *`.

**Note**: Should an error occur while running the query, please see
the AmsterdamUMCdb BigQuery [Frequently Asked Questions](https://github.com/AmsterdamUMC/AmsterdamUMCdb/wiki/bigquery#faq).

In [ ]:
%%bigquery person
SELECT * FROM `amsterdamumcdb.version1_5_0.person`;


## Set the default query job configuration for google-cloud-bigquery client

In [ ]:
from google.cloud import bigquery

# BigQuery requires a separate config to prevent the 'BadRequest: 400 Cannot explicitly modify anonymous table' error message
job_config = bigquery.job.QueryJobConfig()

# sets default client settings by re-using the previously defined config
client = bigquery.Client(project=PROJECT_ID, location=LOCATION, default_query_job_config=def_config)

## 1. Scale of the person table

How many patients does AmsterdamUMCdb contain? Each row in `person` should be a unique individual.

In [ ]:
query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT person_id) AS unique_persons
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.person`
"""

scale_df = client.query(query).to_dataframe()
scale_df

In [ ]:
# Load full person table
query = f"""
SELECT *
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.person`
"""

person_df = client.query(query).to_dataframe()
print(f'Loaded {len(person_df):,} rows, {person_df.shape[1]} columns')
person_df.head()

## 2. Schema sanity — missingness in key fields


AI-assisted (Claude, Anthropic — Claude Opus 4.8): demographic missingness check drafted with AI.

In [ ]:
def missing_stats(df, field):
    n_null = df[field].isna().sum()
    n_zero = (df[field] == 0).sum() if df[field].dtype.kind in 'iuf' else 0
    n_total = len(df)
    return {
        'field': field,
        'missing_or_zero': n_null + n_zero,
        'pct': 100 * (n_null + n_zero) / n_total
    }

rows = [missing_stats(person_df, f) for f in
        ['gender_concept_id', 'year_of_birth', 'race_concept_id', 'ethnicity_concept_id']]
missing_df = pd.DataFrame(rows)
missing_df

AI-assisted (Claude, Anthropic — Claude Opus 4.8): missingness bar plot drafted with AI.

In [ ]:
# Plot missingness
fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#4C72B0' if p < 50 else '#C44E52' for p in missing_df['pct']]
bars = ax.bar(missing_df['field'], missing_df['pct'], color=colors)
ax.set_ylabel('Missing or unknown (%)')
ax.set_title('Missingness in person table required fields')
ax.set_ylim(0, 105)
for bar, pct in zip(bars, missing_df['pct']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
            f'{pct:.1f}%', ha='center', fontsize=10)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

Gender is well recorded only 2.3 % i.e. 465 missing from 20109 patients, every patient has year of birth none missing, race and ethinitcity are missing for all patients. We can use age and gender as features.

## 3. Gender distribution

AI-assisted (Claude, Anthropic — Claude Opus 4.8): gender distribution query drafted with AI.

In [ ]:
query = f"""
SELECT
    COALESCE(c.concept_name, 'Unknown') AS gender,
    COUNT(*) AS n
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.person` p
LEFT JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.concept` c
    ON p.gender_concept_id = c.concept_id
GROUP BY gender
ORDER BY n DESC
"""

gender_df = client.query(query).to_dataframe()
gender_df['pct'] = 100 * gender_df['n'] / gender_df['n'].sum()
gender_df

AI-assisted (Claude, Anthropic — Claude Opus 4.8): gender distribution bar plto debugged with AI.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(gender_df['gender'], gender_df['n'],
              color=['#4C72B0', '#DD8452', '#999999'][:len(gender_df)])
ax.set_ylabel('Number of persons')
ax.set_title(f'Gender distribution (N = {int(gender_df["n"].sum()):,})')
for bar, pct in zip(bars, gender_df['pct']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{pct:.1f}%', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

This table includes 63.6% male and 34.0% female with 2.3 unmapped.

## 4. Year of birth distribution

In [ ]:
yob = person_df['year_of_birth'].dropna()
print(f'Year of birth: min={yob.min()}, max={yob.max()}, '
      f'mean={yob.mean():.1f}, median={yob.median():.0f}')

AI-assisted (Claude, Anthropic — Claude Opus 4.8): Year of Birth distribution query debugged with AI.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(yob, bins=40, color='#4C72B0', edgecolor='white')
ax.set_xlabel('Year of birth')
ax.set_ylabel('Number of persons')
ax.set_title('Year of birth distribution')
ax.axvline(yob.median(), color='red', linestyle='--',
           label=f'Median: {int(yob.median())}')
ax.legend()
plt.tight_layout()
plt.show()

This year of distribution peaks around 1948, most patients are 55-80 years old, these discrete bars indicates de-identification of patients.

## 5. Implausible values (data-quality check)

Flag any rows that are not valid: future birth years, very old patients, or non-adult patients (AmsterdamUMCdb is an adults-only ICU).

AI-assisted (Claude, Anthropic — Claude Opus 4.8): person-table validity checks drafted with AI.

In [ ]:
import pandas as pd
from google.cloud import bigquery
from google.colab import auth

auth.authenticate_user()

# sets *your* project id
PROJECT_ID = "capstoneweaningprediction" #@param {type:"string"}

# sets default dataset for AmsterdamUMCdb
DATASET_PROJECT_ID = 'amsterdamumcdb' #@param {type:"string"}
DATASET_ID = 'version1_5_0' #@param {type:"string"}
LOCATION = 'eu' #@param {type:"string"}

# sets the default query job configuration
def_config = bigquery.job.QueryJobConfig(default_dataset=DATASET_PROJECT_ID + "." + DATASET_ID)

# BigQuery requires a separate config to prevent the 'BadRequest: 400 Cannot explicitly modify anonymous table' error message
job_config = bigquery.job.QueryJobConfig()

# sets default client settings by re-using the previously defined config
client = bigquery.Client(project=PROJECT_ID, location=LOCATION, default_query_job_config=def_config)

# Load full person table — it's small enough to fit in memory
query = f"""
SELECT *
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.person`
"""
person_df = client.query(query).to_dataframe()


CURRENT_YEAR = 2026
checks = {
    'Birth year < 1900': (person_df['year_of_birth'] < 1900).sum(),
    'Birth year > current year': (person_df['year_of_birth'] > CURRENT_YEAR).sum(),
    'Implied age > 110': (CURRENT_YEAR - person_df['year_of_birth'] > 110).sum(),
    'Implied age < 18': (CURRENT_YEAR - person_df['year_of_birth'] < 18).sum(),
    'Gender concept = 0': (person_df['gender_concept_id'] == 0).sum(),
}
pd.Series(checks, name='count')

The only flag here is 465(2.3%) patients with no recorded gender. I will encode them as unknown.

## 6. Age at admission


AI-assisted (Claude, Anthropic — Claude Opus 4.8): Age at admission query debugged with AI.

In [ ]:
query = f"""
SELECT
    p.person_id,
    vo.visit_occurrence_id,
    p.year_of_birth,
    EXTRACT(YEAR FROM vo.visit_start_date) AS admission_year,
    EXTRACT(YEAR FROM vo.visit_start_date) - p.year_of_birth AS age_at_admission
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.person` p
JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.visit_occurrence` vo
    ON p.person_id = vo.person_id
"""

age_df = client.query(query).to_dataframe()
print(f'Loaded {len(age_df):,} admissions')
age_df['age_at_admission'].describe()

AI-assisted (Claude, Anthropic — Claude Opus 4.8): Age at ICU admission bar plot query debuggd with AI.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(age_df['age_at_admission'].dropna(), bins=30,
        color='#55A868', edgecolor='white')
ax.set_xlabel('Age at admission (years)')
ax.set_ylabel('Number of admissions')
ax.set_title('Age at ICU admission')
ax.axvline(age_df['age_at_admission'].median(), color='red', linestyle='--',
           label=f'Median: {age_df["age_at_admission"].median():.0f}')
ax.legend()
plt.tight_layout()
plt.show()

The ICU population is left skewed with a median age of 65 and Inter Quartile Range of 55-65.

## 7. Linkage to visit_occurrence

Every person should have at least one visit.

AI-assisted (Claude, Anthropic — Claude Opus 4.8): debugging — person↔visit integrity check drafted with AI.

In [ ]:
query = f"""
SELECT
    COUNT(DISTINCT p.person_id) AS persons_total,
    COUNT(DISTINCT vo.person_id) AS persons_with_visits,
    COUNT(DISTINCT p.person_id) - COUNT(DISTINCT vo.person_id) AS persons_without_visits
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.person` p
LEFT JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.visit_occurrence` vo
    ON p.person_id = vo.person_id
"""
client.query(query).to_dataframe()

### Distribution of ICU stays per patient

In [ ]:
query = f"""
SELECT
    visits_per_person,
    COUNT(*) AS n_persons
FROM (
    SELECT person_id, COUNT(*) AS visits_per_person
    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.visit_occurrence`
    GROUP BY person_id
) sub
GROUP BY visits_per_person
ORDER BY visits_per_person
"""
visits_df = client.query(query).to_dataframe()
visits_df

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(visits_df['visits_per_person'].astype(str),
       visits_df['n_persons'], color='#8172B2')
ax.set_xlabel('Number of ICU visits per person')
ax.set_ylabel('Number of persons')
ax.set_title('Distribution of ICU stays per patient')
ax.set_yscale('log')
plt.tight_layout()
plt.show()

pct_single = 100 * visits_df.loc[visits_df['visits_per_person'] == 1, 'n_persons'].sum() \
             / visits_df['n_persons'].sum()
print(f'{pct_single:.1f}% of patients have exactly one ICU stay')

The majority fo patients (17800) have single ICU stay, with count dropping for each additional visit on the log scale.

## 8. Admission year distribution

[Zappalà et al. (2025)](https://www.sciencedirect.com/science/article/pii/S0883944125000929) included only admissions from 2010 onwards to reflect modern ventilation practice.

AI-assisted (Claude, Anthropic — Claude Opus 4.8): debugging — admission year distriution bar plot drafted with AI.

In [ ]:
year_counts = age_df['admission_year'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(year_counts.index.astype(int), year_counts.values, color='#937860')
ax.axvline(2009.5, color='red', linestyle='--', label='Zappalà cutoff (2010)')
ax.set_xlabel('Admission year')
ax.set_ylabel('Number of admissions')
ax.set_title('Admissions per year')
ax.legend()
plt.tight_layout()
plt.show()

pre = year_counts[year_counts.index < 2010].sum()
post = year_counts[year_counts.index >= 2010].sum()
print(f'Pre-2010 admissions: {pre:,} ({100*pre/(pre+post):.1f}%)')
print(f'2010+ admissions:    {post:,} ({100*post/(pre+post):.1f}%)')

This plot revels date-binning, admission cluster artificially into 2 large buckets (~9,500 around 2006 and ~13,000 around 2013) confirming AmsterdamUMCdb has aggreagated admission dates into broad windows.

## 9. Combined demographic profile (Table-1 style)

A single summary table mimicking [Zappalà et al. (2025)](https://www.sciencedirect.com/science/article/pii/S0883944125000929) Table 1, before any ventilation filters are applied. We'll re-create this for our final cohort later.

AI-assisted (Claude, Anthropic — Claude Opus 4.8): demographic summary table drafted with AI.

In [ ]:
summary_rows = []

n_persons = person_df['person_id'].nunique()
n_admissions = len(age_df)

summary_rows.append(('Total persons', f'{n_persons:,}'))
summary_rows.append(('Total admissions', f'{n_admissions:,}'))

for _, row in gender_df.iterrows():
    summary_rows.append((f'Gender — {row["gender"]}',
                         f'{row["n"]:,} ({row["pct"]:.1f}%)'))

summary_rows.append(('Median year of birth', f'{int(yob.median())}'))
summary_rows.append(('Median age at admission',
                     f'{age_df["age_at_admission"].median():.0f}'))

# Define age buckets and calculate counts
age_bins = [0, 18, 45, 65, 75, 85, 120]
age_labels = ['<18', '18-44', '45-64', '65-74', '75-84', '85+']
bucket_counts = pd.cut(age_df['age_at_admission'], bins=age_bins, labels=age_labels, right=False).value_counts().sort_index()

for bucket, count in bucket_counts.items():
    pct = 100 * count / bucket_counts.sum()
    summary_rows.append((f'Age {bucket}', f'{count:,} ({pct:.1f}%)'))

summary_df = pd.DataFrame(summary_rows, columns=['Variable', 'Value'])
summary_df